In [0]:
# get the table name from widget (or use default)
dbutils.widgets.text("Database", "sebraepe_dev")
database =  dbutils.widgets.get("Database")

if database not in ["sebraepe_dev", "sebraepe_prod"]:
    raise ValueError("O valor do widget Database deve ser 'sebraepe_dev' ou 'sebraepe_prod' e não pode ser nulo.")

In [0]:
%run /Workspace/sebrae_pe/common/utils/environment

In [0]:
pip install unidecode

  Obtaining dependency information for unidecode from https://files.pythonhosted.org/packages/8f/b7/559f59d57d18b44c6d1250d2eeaa676e028b9c527431f5d0736478a73ba1/Unidecode-1.4.0-py3-none-any.whl.metadata
  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
Using cached Unidecode-1.4.0-py3-none-any.whl (235 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.ml.feature import Tokenizer, StopWordsRemover
import unidecode
import seaborn as sns

In [0]:
# read table
data = spark.read.option("mergeSchema", "true").parquet(
    "/Volumes/sebraepe_dev/landing/raw/tests/testes_sample.parquet")

In [0]:


@F.udf(returnType=StringType())
def normalize_text(text):
    if text is None:
        return ""
    return unidecode.unidecode(text.lower())


def remove_stopwords_spark(df: DataFrame, input_col: str, output_col: str = "clean_text") -> DataFrame:
    """
    Tokenizes and removes stopwords from a text column in a PySpark DataFrame.

    Args:
        df (DataFrame): Input DataFrame.
        input_col (str): Name of the column containing raw text.
        output_col (str): Name of the final cleaned text column (joined tokens).

    Returns:
        DataFrame: Original DataFrame with added columns:
                   - 'tokens': tokenized words
                   - 'filtered_tokens': after stopword removal
                   - output_col: cleaned text as single string
    """
    tokenizer = Tokenizer(inputCol=input_col, outputCol="tokens")
    df_tokenized = tokenizer.transform(df)

    remover = StopWordsRemover(
        inputCol="tokens",
        outputCol="filtered_tokens",
        stopWords=StopWordsRemover.loadDefaultStopWords("portuguese")
    )
    df_filtered = remover.transform(df_tokenized)

    df_cleaned = df_filtered.withColumn(output_col, F.concat_ws(" ", "filtered_tokens"))

    columns_to_drop = ["tokens", "filtered_tokens"]
    df_cleaned = df_cleaned.drop(*columns_to_drop)

    return df_cleaned

In [0]:
data_processed = (data
    # fill null values in specified columns
    .na.fill({
        "descricao": ""
    })
    # normalize and clean text column
    .withColumn("descricao_processed", 
        normalize_text(
            F.trim(F.lower(F.col("descricao")))
        )
    )


)

# to string type
data_processed = (data_processed
    .withColumn("descricao_processed", F.col("descricao_processed").cast(StringType()))
)

# remove special characters
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace(F.col("descricao_processed"), r"[^a-zA-Z0-9\s]", " ")
)

# remove stopwords
data_processed = remove_stopwords_spark(df=data_processed, 
                                            input_col='descricao_processed', 
                                            output_col='descricao_processed')


# Remove números
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace("descricao_processed", "[0-9]", "")
)

# Remove espaços duplicados
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace("descricao_processed", "\\s+", " ")
)



In [0]:
display(data_processed.select("descricao","descricao_processed"))

descricao descricao_processed ENFERMARIA DE GINECOLOGIA 
EVOLUÇÃO DIURNA

DATA: 25/04/2025 
LEITO: 819B
PACIENTE: ERIKA TATTYANY DA SILVA
REGISTRO: 21987565

#HIPÓTESES DIAGNÓSTICAS:
PO HISTERECTOMIA TOTAL DE 24/04/2025

#EM USO:
Sintomáticos

#DADOS DA ENFERMAGEM:
PA: 90x61 | 118x71 | 133x90 mmHg
FC: 62-71 bpm
Tax: sem distermias

#EVOLUÇÃO:
Encontro paciente em enfermaria de ginecologia, acompanhada, em pós-operatório de histerectomia total, sem queixas no momento. Refere episódio de “mal estar/tontura” ao acordar e tentar se levantar, mas com melhora espontânea após poucos minutos, sem novos episódios. Nega febre, náuseas e vômitos. Refere um leve desconforto abdominal, mas tolerável sem dificuldades. Boa aceitação de dieta por via oral, diurese em SVD concentrada e amarelada. Ainda não evacuou, mas com liberação de flatos. Deambulando sem dificuldades.

#EXAME FÍSICO:
GERAL: EGB, consciente e orientada, normocorada, hidratada, anictérica, acianótica, bem perfundida, eupneica
AR: MV+ em AHT, s/ RA | FR: 20 irpm | SpO2 97%
ACV: RCR em 2T, BNF, S/S | FC: 84 bpm | PA: 120x75 mmHg 
ABD: Semigloboso, flácido, doloroso à palpação em baixo ventre, sem VMG, sem sinais de irritação peritoneal. Nota-se um pouco acima da FO, em dobra cutânea, uma lesão linear hiperemiada.
FO: bem coaptada, limpa, seca, sem sinais flogísticos, sem saída de secreção.

#CONDUTA:
Retiro SVD
Estimulo deambulação
Deixo candicorti e cetoconazol para lesão em dobra cutânea. 
Avaliar possibilidade de alta amanhã

 enfermaria ginecologia evolucao diurna data leito b paciente erika tattyany silva registro hipoteses diagnosticas po histerectomia total uso sintomaticos dados enfermagem pa x x x mmhg fc bpm tax distermias evolucao encontro paciente enfermaria ginecologia acompanhada pos operatorio histerectomia total queixas momento refere episodio mal estar tontura acordar tentar levantar melhora espontanea apos poucos minutos novos episodios nega febre nauseas vomitos refere leve desconforto abdominal toleravel dificuldades boa aceitacao dieta via oral diurese svd concentrada amarelada ainda nao evacuou liberacao flatos deambulando dificuldades exame fisico geral egb consciente orientada normocorada hidratada anicterica acianotica bem perfundida eupneica ar mv aht s ra fr irpm spo acv rcr t bnf s s fc bpm pa x mmhg abd semigloboso flacido doloroso palpacao baixo ventre vmg sinais irritacao peritoneal nota pouco acima fo dobra cutanea lesao linear hiperemiada fo bem coaptada limpa seca sinais flogisticos saida secrecao conduta retiro svd estimulo deambulacao deixo candicorti cetoconazol lesao dobra cutanea avaliar possibilidade alta amanha 10/03/24 - (10:20) - RN estável, ativo/reativo; corado; admitido na UTIN proveniente do COB, sendo transportado em incubadora de transporte com suporte ventilatório do BABY PUFF; colocado sob CPAP, gemente (leve); TSC/TIC leve, PV superficial/irregular; hipotérmico; não apresenta secreção em VAS no momento. Foi observado pé direito cianótico (parto pélvico ?). Sinais Vitais: FR 46-50ipm; FC 136bpm; Spo2 96%

AP: RR +, s/ ruídos adventícios.

CPAP: FiO2 23%, PEEP 6; FLUXO 8 L/ min ; PMVA 6
S/F: 4,17


Conduta: Avaliação, Monitorização cardiorespiratória; DRR ; Mantidos parâmetros ventilatórios; Ajuste de posicionamento; Vigilância respiratória.






10/03/24 - (14:00) - RN estável, ativo/reativo; corado; sob CPAP, superou a gemência ; TSC/TIC leve, PV superficial/irregular; segue hipotérmico; não apresenta secreção em VAS no momento. Sinais Vitais: FR 46-50ipm; FC 136bpm; Spo2 96%

AP: RR +, s/ ruídos adventícios.

CPAP: FiO2 23%, PEEP 6; FLUXO 8 L/ min ; PMVA 6
S/F: 4,17


Conduta: Avaliação, Monitorização cardiorespiratória; DRR; Mantidos parâmetros ventilatórios; Ajuste de posicionamento; Vigilância respiratória

 rn estavel ativo reativo corado admitido utin proveniente cob sendo transportado incubadora transporte suporte ventilatorio baby puff colocado sob cpap gem

In [0]:
# converte as strings para timestamp
data_processed = data_processed.withColumn("data_saida", F.to_timestamp("data_saida")) 

🔹 Ambiente definido: dev
🔹 Catálogo ativo: sebraepe_dev


In [0]:
# filter cases above august 2022
data_processed = data_processed.filter(F.col("data_saida") >= F.lit("2022-08-01"))

In [0]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("data_saida")).alias("min_m"),
    F.date_trunc("month", F.max("data_saida")).alias("max_m"),
)

ref_dates_data = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_data.collect()]
len(ref_dates)

44

In [0]:
# define dataframe para incorporar dados ao cursor
evolucao = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data_processed.filter(
        (F.col("data_saida") < F.lit(ref_ts)) &
        (F.col("data_saida") >= F.add_months(F.lit(ref_ts), -12))
    )

    # concatenar as evolucoes do paciente
    evolucao_ref_date = data_filtered.groupBy("prontuario").agg(
        F.concat_ws(" ", F.collect_list("descricao_processed")).alias("descricao_unica")
    )

    evolucao_ref_date = evolucao_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    evolucao = evolucao.unionByName(evolucao_ref_date, allowMissingColumns=True)

    print(ref_ts)

2021-10-01 00:00:00
2021-11-01 00:00:00
2021-12-01 00:00:00
2022-01-01 00:00:00
2022-02-01 00:00:00
2022-03-01 00:00:00
2022-04-01 00:00:00
2022-05-01 00:00:00
2022-06-01 00:00:00
2022-07-01 00:00:00
2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00
2025-04-01 00:00:00
2025-05-01 00:00:00


In [0]:
display(evolucao)

prontuario descricao_unica date_ref 10103273 paciente pre operatorio nefrectomia total nefrolitiase direita transplantada renal ha anos doador vivo paciente hipertensa diabetica portadora parkinson passado tvp nega alergias evolui estado geral regular consciente orientada eutimica normocorada eupneica ar ambiente sono repouso prejudicados quadro insonia dieta via oral zero ate procedimento cirurgico cabeca pescoco alteracoes torax simetrico expansivo bilateralmente abdome semi globoso flacido indolor diurese evacuacoes presentes porem paciente portadora constipacao cronica extremidades perfundidas hidratadas edemas lesoes apresenta tremores continuos quadro parkinson deambula auxilio uso avp mse sinais flogisticos conduta orientada quanto cuidados pre operatorios obs suspenso procedimento cirurgico falta vaga uti paciente alta hospitalar 2022-09-01T00:00:00Z 11395027 paciente nao colaborativo expressa indignacao ter sido informado iria alta hoje medico informa nao iria paciente nao expressa grandes limitacoes fisicas queixas respiratorias retorno momento oportuno paciente consciente orientado eupneico re aa afebril toque spo fc bpm pa xmmhg realizando atb terapia relata sentir moleza nao deseja realizar esforco respeito vontade paciente evolucao ccp dpo paratireoidectomia total autoimplante hiperparatireoidismo secundario evolucao paciente evolui estavel intercorrencias episodios dispneia disfonia sinais hipocalcemia relatados tremores parestesias caimbras refere dor leve local fo melhora dor ossea previa apos cirurgia ef fo bom aspecto abaulamentos dreno bem posicionado controles pas pad fc tax hgt debito dreno ml conduta refaco curativo orientacao liberar dieta via oral branda renal cronico dialitico manter dreno avaliar debito h 2022-09-01T00:00:00Z 11413929 enfermaria geriatria paciente iraci albuquerque santos idade anos prontuario enfermaria a lista problemas geriatricos sd fragilidade cfs previo internamento alto risco queda obesidade sarcopenica imc kg m desnutricao clinicos sepse foco urinario infeccao corrente sanguinea tratamento hemoculturas positivas coli amostras sd emetica aguda hipocalemia constipacao sec uso cronico opioide dor abdominal sec p disfagia drge disfagia sarcopenica diverticulo zenker globus hystericus superado paraparesia flacida crural arreflexa retencao urinaria constipacao sind compressao radicular uso cronico opioide neoplasia mama esquerda mastectomia esquerda esvaziamento total rt ha anos hiponatremia sintomatica desidratacao siadh insuficiencia adrenal superado transtorno humor depressivo ansioso grave tratamento has dm ii dor lombossacra sec compressao radicular osteoartrose coluna lombossacra doenca diverticular colons candidiase oral tratada uso dipirona g h codeina mg h sos hnf profilatica bisacodil mg dia omeprazol mg dia simeticona mg h domperidona mg dia plasil mg h lactulose ml dia duloxetina mg dia d aumento dose vitamina d semanal inicio antibioticoterapia meronem g h d teicoplanina mg h d fez uso fluconazol d mg partir d mg dia nistatina suspensao oral d tazocin dispositivos avc vjie svd evolucao paciente segue dormindo leito filho relata baixa aceitacao dieta via oral evacuar episodios febris dor abdominal pas pad fc fr tax hgt exame fisico eg grave consciente pouco interativa hipocorada eupneica anicterica afebril ar mv aht reduzido bases fr ipm acv rci t bnf s ss abd globoso flacido depressivel indolor palpacao profunda difusamente sinais irritacao peritoneal rha hipoativos edema mmii simetrico indolor dermatite ocre tec seg empastamento panturrilha direita exames laboratoriais hb ht leuco mielo metamielo bast seg lt mono plq cr fa ggt k cl mg p pcr tgo tgp ur su ph leuco nitrito sangue piocitos incontaveis hemacias varias bacterias varias hemocultura amostras positivo hm hm escherichia coli resistencia ampicilina ampicilina sulbactam ceftriaxona cefuroxima ciprofloxacina gentamicina urocultura nova amostra urina contaminada hb ht leuco bast seg plq bt bd cr ca k cl mg pcr inr

In [0]:
#display(data.select())

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1765061787423952>, line 13
      7 data_filtered = data.filter(
      8     (F.col("data_saida") < F.lit(ref_ts)) &
      9     (F.col("data_saida") >= F.add_months(F.lit(ref_ts), -12))
     10 )
     12 # concatenar as evolucoes do paciente
---> 13 evolucao_ref_date = data_filtered.groupBy("prontuario").agg(
     14     F.concat_ws(" ", F.collect_list("descricao_processed")).alias("descricao_unica")
     15 )
     17 evolucao_ref_date = evolucao_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))
     19 evolucao = evolucao.unionByName(evolucao_ref_date, allowMissingColumns=True)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_s

In [0]:
### Feature Engin

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1765061787423952>, line 13
      7 data_filtered = data.filter(
      8     (F.col("data_saida") < F.lit(ref_ts)) &
      9     (F.col("data_saida") >= F.add_months(F.lit(ref_ts), -12))
     10 )
     12 # concatenar as evolucoes do paciente
---> 13 evolucao_ref_date = data_filtered.groupBy("prontuario").agg(
     14     F.concat_ws(" ", F.collect_list("descricao_processed")).alias("descricao_unica")
     15 )
     17 evolucao_ref_date = evolucao_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))
     19 evolucao = evolucao.unionByName(evolucao_ref_date, allowMissingColumns=True)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_s